In [ ]:

import math
import matplotlib.pyplot as plt

# Global variables
pruned_nodes = []
visited_nodes = []


# Alpha-Beta Pruning Algorithm
def alpha_beta(node, depth, alpha, beta, maximizing, path="Root"):

    if isinstance(node, (int, float)):
        visited_nodes.append(path)
        print(f"Visited {path}: {node}")
        return node

    if maximizing:
        best = -math.inf

        for i, child in enumerate(node):
            child_path = f"{path}.{i + 1}"

            value = alpha_beta(
                child, depth + 1, alpha, beta,
                False, child_path
            )

            best = max(best, value)
            alpha = max(alpha, best)

            print(f"MAX at {path}: Alpha = {alpha}, Beta = {beta}")

            if alpha >= beta:
                for j in range(i + 1, len(node)):
                    pruned_nodes.append(f"{path}.{j + 1}")

                if i + 1 < len(node):
                    print(f"PRUNING at {path}!")
                break

        return best

    else:
        best = math.inf

        for i, child in enumerate(node):
            child_path = f"{path}.{i + 1}"

            value = alpha_beta(
                child, depth + 1, alpha, beta,
                True, child_path
            )

            best = min(best, value)
            beta = min(beta, best)

            print(f"MIN at {path}: Alpha = {alpha}, Beta = {beta}")

            if alpha >= beta:
                for j in range(i + 1, len(node)):
                    pruned_nodes.append(f"{path}.{j + 1}")

                if i + 1 < len(node):
                    print(f"PRUNING at {path}!")
                break

        return best


# Display Game Tree in Terminal
def print_tree(node, level=0, label="Root"):
    print("    " * level + f"{label}: {node if isinstance(node, (int, float)) else ('MAX' if level % 2 == 0 else 'MIN')}")

    if isinstance(node, list):
        for i, child in enumerate(node):
            print_tree(child, level + 1, f"Child {i + 1}")


# Get all leaf values inside a subtree
def get_leaves(node):
    if isinstance(node, (int, float)):
        return [node]

    leaves = []
    for child in node:
        leaves.extend(get_leaves(child))
    return leaves


# Draw Graphical Game Tree
def draw_tree(tree):

    fig, ax = plt.subplots(figsize=(12, 7))

    def is_pruned(path):
        return any(
            path == p or path.startswith(p + ".")
            for p in pruned_nodes
        )

    def draw(node, x, y, width, level, path):

        pruned = is_pruned(path)

        if isinstance(node, (int, float)):
            color = "red" if pruned else "lightgreen"
            label = str(node)
        else:
            color = "red" if pruned else "skyblue"
            label = "MAX" if level % 2 == 0 else "MIN"

        ax.text(
            x, y, label,
            ha="center", va="center",
            fontsize=11,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor=color,
                edgecolor="black"
            )
        )

        if isinstance(node, list):
            count = len(node)

            for i, child in enumerate(node):
                child_x = x - width / 2 + (i + 0.5) * width / count
                child_y = y - 1
                child_path = f"{path}.{i + 1}"

                ax.plot(
                    [x, child_x],
                    [y - 0.15, child_y + 0.15],
                    color="red" if is_pruned(child_path) else "black"
                )

                draw(
                    child, child_x, child_y,
                    width / count, level + 1,
                    child_path
                )

    draw(tree, 0, 0, 12, 0, "Root")

    ax.set_xlim(-7, 7)
    ax.axis("off")
    ax.set_title("Alpha-Beta Pruning Game Tree")

    plt.savefig("alpha_beta_tree.png", bbox_inches="tight")
    plt.show()


# Main Program
print("\n========== ALPHA-BETA PRUNING ==========")

print("\n1. Enter your own game tree")
print("2. Use default game tree from PPT")

choice = input("\nEnter your choice (1 or 2): ")

if choice == "1":
    print("\nEnter leaf values separated by spaces.")
    print("Example: 10 9 14 18 5 4 50 3")
    values = list(map(int, input("Enter leaf values: ").split()))

    if len(values) != 8:
        print("Please enter exactly 8 values.")
        exit()

    tree = [
        [[values[0], values[1]], [values[2], values[3]]],
        [[values[4], values[5]], [values[6], values[7]]]
    ]

else:
    tree = [
        [[10, 9], [14, 18]],
        [[5, 4], [50, 3]]
    ]

print("\n========== GAME TREE ==========")
print_tree(tree)

print("\n========== ALPHA-BETA EVALUATION ==========")

result = alpha_beta(
    tree, 0,
    -math.inf, math.inf,
    True
)

print("\n========== FINAL RESULT ==========")
print("Final MAX Root Value:", result)

all_pruned = []

def collect_pruned(node, path="Root"):
    if any(path == p or path.startswith(p + ".") for p in pruned_nodes):
        all_pruned.extend(get_leaves(node))
        return

    if isinstance(node, list):
        for i, child in enumerate(node):
            collect_pruned(child, f"{path}.{i + 1}")

collect_pruned(tree)

print("Pruned Leaf Nodes:", all_pruned)

print("\nDisplaying Game Tree...")
draw_tree(tree)

print("\nTree image saved as alpha_beta_tree.png")
